# Divide and Conquer
In the previous notebook we looked at two different methods for evaluating polynomials. By counting the number of elementary operations (elops) we found that Horner's rule is more efficient. One reason the naive method is slow is that raising a number to the $n$-th power requires $n-1$ multiplications. In this notebook we will revisit this issue. It turns out that there it is possible to compute $x^n$ with significantly fewer multiplications. The algorithm we will develop for doing this is an example of a so-called Divide-and-Conquer method. It turns out that the Divide-and-Conquer approach is a more general concept that we will come across again in the next lectures.

<hr style="height: 2px">

### What you will learn
In this notebook we will cover the following topics:

* Computing powers more efficiently
* The FastPower algorithm
* Counting operations
* Divide-and-Conquer algorithms

<hr style="height: 2px">

A technical proof included at the very end of the notebook makes one of the mathematical ideas presented here more rigorous. Although this proof is not examinable, it will help you with understanding the material and you are encouraged to work through it.

*&#169; Eike Mueller, University of Bath 2019-2023. These lecture notes are copyright of Eike Mueller, University of Bath. They are provided exclusively for educational purposes at the University and are to be downloaded or copied for your private study only. Further distribution, e.g. by upload to external repositories, is prohibited.*

## Computing powers more efficiently
Recall that raising a number $x$ to the $n$-th power with the 'naive' method requires $n-1$ elops since

$$
x^n = \underbrace{x\cdot x\cdot x \cdots x\cdot x}_{n-1\;\text{multiplications}}
$$

More explicitly, we write

$$
E_{\text{naive_power}}(n) = n-1.
$$

So for example computing $x^8$ requires $8-1=7$ multiplications.

However, there is a better way of doing this:

1. Set $z_1 = x$ (no multiplication)
2. Compute $z_2 = z_1\cdot z_1$ (1 multiplication)
3. Compute $z_4 = z_2\cdot z_2$ (1 multiplication)
4. Compute $z_8=z_4\cdot z_4$ (1 multiplication)

This only requires *three* (instead of seven) multiplications!

This works since

$$
\begin{aligned}
z_8&=z_4\cdot z_4\\
&= (z_2\cdot z_2) \cdot (z_2\cdot z_2)\\
&= ((z_1\cdot z_1)\cdot (z_1\cdot z_1)) \cdot ((z_1\cdot z_1)\cdot (z_1\cdot z_1))\\
&= ((x\cdot x)\cdot (x\cdot x)) \cdot ((x\cdot x)\cdot (x\cdot x))\\
&= x^8
\end{aligned}
$$

which is the result we were looking for. Following the same idea, it can be shown that $x^{16}$ can be computed with 4 multiplications, $x^{32}$ with 5 multiplications and so on. As we can see from the example above, the key step of the algorithm is to compute 

$$z_j = z_{j/2}\cdot z_{j/2}$$

for $j=n,n/2,n/4,\dots$ with $z_1 = x$. In general, if $n=2^k$ for an integer $k\in\mathbb{N}$, we can compute $x^n$ this way, requiring only $k=\log_2(n)$ multiplications.

An alternative way of looking at this is to observe that if $n$ is an even integer then

$$
x^n = (x\cdot x)^{n/2}
$$

In other words, we can reduce the computation of $x^n$ (*''$x$ raised to the power $n$''*) to the computation of $(x\cdot x)^{n/2}$ (*''$x\cdot x$ raised to the power $n/2$''*) with obviously $x^1=x$. This leads to the following algorithm, which is expressed as a recursive function for computing $x^n$:

### Function $\text{fast_binary_power}(x,n)$
**Input**: value $x$, Integer power $n$ such that $n=2^k$ with $k\in \mathbb{N}$;
**Output** $x^{n}$
1. **if** $n==1$ **then**
1. $~~~~$ **return** $x$
1. **else**
1. $~~~~$ **return** $\text{fast_binary_power}(x\cdot x,n/2)$
1. **end if**
1. **return** $p$

A natural question is now how many elops we need to compute $x^n$ with this algorithm for a given power $n$. To work this out, observe that in each recursive call the argument $n$ is divided by a factor $2$, i.e. we have

$$
\text{fast_binary_power}(x,n) \rightarrow \text{fast_binary_power}(x^2,n/2) \rightarrow \text{fast_binary_power}(x^4,n/4) \rightarrow \dots \rightarrow \text{fast_binary_power}(x^n,1)
$$

If $n=2^k$, then there are $k$ recursive calls of the function $\text{fast_binary_power}$. In each recursive call we do one multiplication, since $x$ is multiplied by itself before it is passed as an argument to $\text{fast_binary_power}$. We conclude that the total number of elops is 

$$
E_{\text{fast_binary_power}}(n) = k = \log_2 n
$$

Here is a Python implementation of the pseudocode above:

In [2]:
def fast_binary_power(x,n):
    '''Compute x^n using recursion
    
    input:
      * x value to raise to the n-th power
      * n power n=2^k (NOTE THAT n HAS TO BE A POWER OF TWO)
    output:
      x^n
    '''
    if n==1:
        return x
    else:
        return fast_binary_power(x*x,n//2) # Use x^n = (x*x)^{n/2}

Let us use this to compute $3^4 = 3^{2^2} = 81$ and $5^{16} = 5^{2^4} =152587890625$:

In [3]:
fast_binary_power(3,4)

81

In [4]:
fast_binary_power(5,16)

152587890625

If $n$ is not a power of two we clearly get garbage, for example:

In [5]:
fast_binary_power(4,3)

16

but this should be $4^3=64$.

Obviously, the drawback of this algorithm is that it can only be used to raise $x$ to some power $n$ which is a power of two, i.e. $n=2^k$ with $k\in\mathbb{N}$. To overcome this limitation, observe that:

$$
x^n = \begin{cases}
(x\cdot x)^{n/2} & \text{if $n\ge 2$ is even}\\
x\cdot (x\cdot x)^{(n-1)/2} & \text{if $n$ is odd}
\end{cases}
$$

with $x^0 = 1$. It also makes sense to treat the special case $x=1$ separately, since obviously $x^1=x$.

For example, we have that $x^7 = x\cdot (x\cdot x)^{(7-1)/2}$. This allows us to generalise the $\text{fast_binary_power}$ function to arbitrary powers $n\in \mathbb{N}$ by defining a function $\text{fast_power()}$ which satisfies:

$$
\text{fast_power}(x,n) = \begin{cases}
\text{fast_power}(x\cdot x,n/2) & \text{if $n\ge 2$ is even}\\
x\cdot\text{fast_power}(x\cdot x,(n-1)/2) & \text{if $n\ge 3$ is odd}\\
x & \text{if $n=1$}\\
1 & \text{if $n=0$}.
\end{cases}
$$

Here is the pseudocode for this function:

### Function $\text{fast_power}(x,n)$
**Input**: value $x$, Integer power $n$;
**Output** $x^{n}$
1. **if** $n==0$ **then**
1. $~~~~$ **return** $1$
1. **else if** $n==1$ **then**
1. $~~~~$ **return** $x$
1. **else if** $n$ is even **then**
1. $~~~~$ **return** $\text{fast_power}(x\cdot x,n/2)$
1. **else**
1. $~~~~$ **return** $x\cdot \text{fast_power}(x \cdot x,(n-1)/2)$
1. **end if**

Assume, for example that we want to compute $x^{14}$. This will lead to the following sequence of recursive function calls:

$$
\text{fast_power}(x,14) \rightarrow \text{fast_power}(x^2,7)\rightarrow x^2\cdot \text{fast_power}(x^4,3) \rightarrow x^2\cdot \left(x^4 \cdot \text{fast_power}(x^8,1) \right) 
$$

You will implement a variant of the $\text{fast_power}$ algorithm in Tickable 6.

### Counting operations
Again, we can estimate the number of elops $E_{\text{fast_power}}(n)$ required to execute this algorithm for a given value of $n$. To obtain an upper bound on $E_{\text{fast_power}}(n)$, first consider $n\ge1$ and observe that

1. in each call to $\text{fast_power}$ we need to do at most two multiplications. This can be seen as follows:
  * We always compute $x\cdot x$ before passing this number as an argument to $\text{fast_power}$ in lines 6 and 8, that is one multiplication.
  * If $n$ is odd and larger than $1$, we carry out an additional multiplication with $x$ in line 8.
  * If $n=1$ we do not need to do any multiplication.
2. In each recursive call the power $n$ is reduced at least by a factor of two:
  * In line 6, $\text{fast_power}()$ is called with $\frac{n}{2}$
  * In line 8, $\text{fast_power}()$ is called with $\frac{n-1}{2}\le \frac{n}{2}$
3. The recursion stops if $n=1$.

From 2. and 3. we infer that the number of recursive calls to $\text{fast_power}()$ (in other words, the number of times lines 6 or 8 of the algorithm are executed) can not be larger than $\log_2(n)$. A mathematically more rigorous proof of this observation can be found at the end of this notebook. The total number of elops can then be bounded as follows:

$$
\begin{aligned}
E_{\text{fast_power}}(n) &\le (\text{maximal number of elops per call}) \times (\text{maximal number of recursive calls})\\
&= 2 \log_2 n \qquad\text{for $n\ge 1$}
\end{aligned}
$$

If $n=0$, we do not need to multiply anything, so $E_{\text{fast_power}}(0)=0$.

We deduce that

$$
E_{\text{fast_power}}(n) \le 2\log_2 n \qquad\text{for $n\ge 1$}
$$

Recall that the number of elops in the naive algorithm is $E_{\text{naive_power}}(n) = n-1$, so

$$
E_{\text{fast_power}}(n) < E_{\text{naive_power}}(n)\qquad\text{for $n\ge 7$.}
$$

The saving is dramatic for high powers $n$: Computing $x^{10000}$ requires $9999$ multiplications with the naive algorithm, but fewer than $27$ multiplications with $\text{fast_power}$.

## Divide and Conquer
The $\text{fast_power}$ method is an example of a so-called *Divide and Conquer* algorithm. The general idea behind those methods is to take a problem of a certain size and split it into two smaller subproblems which have approximately half the size. In our case the problem of raising some number $x$ to the power $n$ is reduced to the problem of raising some other number to the power $n/2$ or $(n-1)/2$ ($\approx n/2$ for large values of $n$).

Many efficient algorithms in computer science are based on similar ideas. We will meet other examples later in the course when we look at sorting algorithms. Although this not always the case, the number of elops in Divide and Conquer algorithms can often be bounded by an expression of the form $C n^\alpha \log_2(n)$ where $C$ is some constant and $\alpha$ is an integer number. For the $\text{fast_power}$ method we found that $E_{\text{fast_power}}(n)\le 2\log_2(n)$, i.e. $C=2$ and $\alpha=0$.

## Check your understanding
### Exercise 1
Exactly many elops are required to compute $x^7$ with the $\text{fast_power}$ method?

In total, four multiplications are required.

We have that
$$
x^7 = \text{fast_power}(x,7) = x\cdot \text{fast_power}(x_2,3) = x\cdot x_2 \cdot \text{fast_power}(x_4,1) = x\cdot x_2 \cdot x_4
$$

Where $x_2=x^2$ and $x_4 = x_2^2 = x^4$.

We need one multiplication to compute $x_2$ from $x$ and then another multiplication to compute $x_4$ from $x_2$, in addition to the two multiplications in the expression $x\cdot x_2 \cdot x_4$.

Note that the naive method would require six multiplications.

### Exercise 2
Exactly many elops are required to compute $x^{8}$ with the $\text{fast_power}$ method?

We need three multiplication in total. Again, we can write this as

$$
x^8 = \text{fast_power}(x,8) = \text{fast_power}(x_2,4) = \text{fast_power}(x_4,2) = \text{fast_power}(x_8,1)
$$

With $x_2=x^2$, $x_4=x_2^2=x^4$, $x_8=x_4^2=x^8$. We need one multiplication to compute $x_2$ from $x$, then another one to compute $x_4$ from $x_2$ and finally one to compute $x_8$ from $x_4$. Note that for the naive method we would require seven multiplications and that the number of elops required to compute $x^8$ with the $\text{fast_power}()$ method is smaller than the number of elops required to compute $x^7$ with $\text{fast_power}()$.

### Exercise 3
Exactly many elops are required to compute $x^{11}$ with the $\text{fast_power}$ method?

Proceeding as above we find that

$$
x^{11} = \text{fast_power}(x,11) = x\cdot \text{fast_power}(x_2,5) = x\cdot x_2\cdot \text{fast_power}(x_4,2) = x\cdot x_2\cdot \text{fast_power}(x_8,1) = x\cdot x_2\cdot x_8
$$

With $x_2$, $x_4$ and $x_8$ defined as above. The total number of multiplications required to compute $x^{11}$ with the $\text{fast_power}()$ method is therefore five.

### Exercise 4
Exactly many elops are required to compute $x^{4095}$ with the $\text{fast_power}$ method?
Hint: $4095=2^{12}-1$.

Computing $4095$ with the $\text{fast_power}$ method requires $22$ elops.

To see this, let $\tilde{E}(k)$ be the number of elops required to compute $x^{n}$ if $n=2^k-1$. We can compute $\tilde{E}(k)$ recursively, starting from $\tilde{E}(1)=0$. For this, observe that for $k\ge 2$ the number $n=2^k-1$ is an odd number larger or equal to three, so

$$
\text{fast_power}(x,2^k-1) = x\cdot \text{fast_power}(x\cdot x,\frac{1}{2}(2^k-2)) = x\cdot \text{fast_power}(x\cdot x,2^{k-1}-1).
$$

Hence, each recursive call adds two more multiplications and thus

$$
\tilde{E}(k)=2+\tilde{E}(k-1) = 2+2+\tilde{E}(k-2) = \underbrace{2+2+\dots+2}_{k-1\;\text{times}}E(1) = 2(k-1).
$$

For $n=4095=2^{12}-1$ we find that

$$
E(n) = \tilde{E}(12) = 2\cdot 11 = 22.
$$

## Detailed derivation of the number of recursive calls in $\text{fast_power}$ (not examinable)
Let $\nu(n)$ be the number of recursive calls in $\text{fast_power}$. In this section we will show that $\log_2(n)-1<\nu(n)\le \log_2(n)$ for $n\ge 1$ in a mathematically rigorous way.

For this, we first need to prove a Theorem about the floor-function. Recall that for any real number $x$ the floor of $x$, denoted by $\lfloor x\rfloor=\max_{n\in\mathbb{Z}}\{n\le x\}$ is the largest integer that is less than or equal to $x$. For example

$$
\lfloor 1.1\rfloor = 1,\quad \lfloor -1.1\rfloor=-2,\quad\lfloor 4\rfloor  = 4,\quad\lfloor 3.999\rfloor = 3
$$

More specifically, for every $x\in\mathbb{R}$ there exists a unique $F = \lfloor x\rfloor\in \mathbb{Z}$ and $0\le \delta < 1$ such that $x=F+\delta$.

We now have the following

### Theorem
*For an real number $x\in\mathbb{R}$*

$$
\lfloor \frac{\lfloor x\rfloor}{2}\rfloor = \lfloor\frac{x}{2}\rfloor
$$

#### Proof
First consider the case where $\lfloor x\rfloor$ is even, i.e. $\lfloor x\rfloor=2m$ with $m\in\mathbb{Z}$. In this case we can write $x=2m+\delta$ with $0\le \delta<1$ (note that $\delta$ is *strictly* less than $1$). Using the definition of the floor, we then have

$$
\begin{aligned}
\lfloor x\rfloor = 2m\;\Rightarrow\;\frac{\lfloor x\rfloor}{2} = m\;\Rightarrow\;\lfloor\frac{\lfloor x\rfloor}{2}\rfloor = m\\
\frac{x}{2} = m +\frac{\delta}{2}\;\Rightarrow\;\lfloor\frac{x}{2}\rfloor = m
\end{aligned}
$$

On the other hand, if $\lfloor x\rfloor$ is odd, then $x=2m+1+\delta$ with $m\in\mathbb{Z}$ and $0\le \delta<1$. In this case

$$
\begin{aligned}
\lfloor x\rfloor = 2m+1\;\Rightarrow\;\frac{\lfloor x\rfloor}{2} = m+\frac{1}{2}\;\Rightarrow\;\lfloor\frac{\lfloor x\rfloor}{2}\rfloor = m\\
\frac{x}{2} = m +\underbrace{\frac{1}{2}+\frac{\delta}{2}}_{<1}\;\Rightarrow\;\lfloor\frac{x}{2}\rfloor = m
\end{aligned}
$$

This completes the proof.

Next, observe that if $n$ is even, then $\lfloor \frac{n}{2}\rfloor = \frac{n}{2}$. If $n$ is odd, then $\lfloor\frac{n}{2}\rfloor = \frac{n-1}{2}$. Hence, we could replace line 6 in the $\text{fast_power}$ algorithm by

$~~~~$ **return** $\text{fast_power}(x\cdot x,\lfloor n/2\rfloor)$

and line 8 by

$~~~~$ **return** $x\cdot \text{fast_power}(x\cdot x,\lfloor n/2\rfloor)$

In other words, when the method calls itself recursively, it always uses $\lfloor n/2\rfloor$ as the second argument. From this we deduce that

$$
\nu(n) = 1 + \nu\left(\lfloor \frac{n}{2}\rfloor\right).
$$

We also have that $\nu(1)=0\le \log_2(1)$. Next consider $n\ge 2$. In this case there exists a unique integer $k\ge 1$ such that

$$
2^k \le n < 2^{k+1}.
$$

Observe that $1\le \frac{n}{2^{k}}< 2$, so $\lfloor\frac{n}{2^{k}}\rfloor = 1$ and also (since the logarithm is an increasing function) $k\le \log_2 n$.

Repeatedly applying the identity $\nu(n) = 1 + \nu\left(\lfloor \frac{n}{2}\rfloor\right)$ and using the result $\lfloor \frac{\lfloor x\rfloor}{2}\rfloor = \lfloor\frac{x}{2}\rfloor$ from the above theorem we find that

$$
\begin{aligned}
\nu(n) &= 1 + \nu\left(\lfloor\frac{n}{2}\rfloor\right)\\
&= 1 + 1 + \nu\left(\lfloor\frac{\lfloor\frac{n}{2}\rfloor}{2}\rfloor\right) = 2 + \nu\left(\lfloor\frac{n}{4}\rfloor\right)\\
&= 2 + 1 + \nu\left(\lfloor\frac{\lfloor\frac{n}{4}\rfloor}{2}\rfloor\right) = 3 + \nu\left(\lfloor\frac{n}{8}\rfloor\right)\\
&= 3 + 1 + \nu\left(\lfloor\frac{\lfloor\frac{n}{8}\rfloor}{2}\rfloor\right) = 4 + \nu\left(\lfloor\frac{n}{16}\rfloor\right)\\
&=\quad\dots\\
&= k + \nu\left(\lfloor\frac{n}{2^k}\rfloor\right) = k + \nu\left(1\right)\\
&= k.
\end{aligned}
$$

Hence $\nu(n)=k$ and therefore as required

$$
\log_2(n)-1 < \nu(n) \le \log_2(n),
$$

which completes the proof.